In [47]:
import numpy as np
from typing import Tuple
from numpy.lib.scimath import sqrt as csqrt
from scipy.special import gamma, psi
import math

In [41]:
def BOsborne(signal, xi: complex, n: int) -> Tuple[complex, complex]:
    T1 = signal.T1()
    T2 = signal.T2()
    dt = (T2 - T1) / n
    i = 1j

    f1 = np.exp(-i * xi * T1)
    f2 = 0.0 + 0.0j

    for j in range(1, n + 1):
        t = T1 + (j - 0.5) * dt
        q = signal.q(t)
        abs_q = abs(q)
        tmp1, tmp2 = f1, f2
        k = csqrt(-abs_q ** 2 - xi ** 2)

        cosh_kdt = np.cosh(k * dt)
        sinh_kdt = np.sinh(k * dt)

        f1 = (cosh_kdt - (i * xi / k) * sinh_kdt) * tmp1 + tmp2 * sinh_kdt * q / k
        f2 = -tmp1 * np.conj(q) * sinh_kdt / k + (cosh_kdt + (i * xi / k) * sinh_kdt) * tmp2

    a = f1 * np.exp(i * xi * T2)
    b = f2 * np.exp(-i * xi * T2)
    return a, b


In [42]:
def BOsbornePrime(signal, xi: complex, n: int) -> complex:
    T1 = signal.T1()
    T2 = signal.T2()
    dt = (T2 - T1) / n
    i = 1j

    f1 = np.exp(-i * xi * T1)
    f2 = 0.0 + 0.0j
    fPrime1 = -i * T1 * np.exp(-i * xi * T1)
    fPrime2 = 0.0 + 0.0j

    for j in range(1, n + 1):
        t = T1 + (j - 0.5) * dt
        q = signal.q(t)
        abs_q = abs(q)
        tmp1, tmp2 = f1, f2
        tmpPrime1, tmpPrime2 = fPrime1, fPrime2
        k = csqrt(-abs_q ** 2 - xi ** 2)
        ik = i / k

        cosh_kdt = np.cosh(k * dt)
        sinh_kdt = np.sinh(k * dt)

        tPrime11 = (
            i * xi ** 2 * dt * cosh_kdt / (k ** 2)
            - (xi * dt / k + ik + i * xi ** 2 / (k ** 3)) * sinh_kdt
        )
        tPrime12 = xi * sinh_kdt * q / (k ** 3) - xi * dt * cosh_kdt * q / (k ** 2)
        tPrime21 = (
            -xi * np.conj(q) * sinh_kdt / (k ** 3)
            + xi * dt * np.conj(q) * cosh_kdt / (k ** 2)
        )
        tPrime22 = (
            -i * xi ** 2 * dt * cosh_kdt / (k ** 2)
            + (-xi * dt / k + ik + i * xi ** 2 / (k ** 3)) * sinh_kdt
        )

        t11 = cosh_kdt - (i * xi / k) * sinh_kdt
        t12 = sinh_kdt * q / k
        t21 = -np.conj(q) * sinh_kdt / k
        t22 = cosh_kdt + (i * xi / k) * sinh_kdt

        fPrime1 = tPrime11 * tmp1 + tPrime12 * tmp2 + t11 * tmpPrime1 + t12 * tmpPrime2
        fPrime2 = tPrime21 * tmp1 + tPrime22 * tmp2 + t21 * tmpPrime1 + t22 * tmpPrime2

        f1 = t11 * tmp1 + t12 * tmp2
        f2 = t21 * tmp1 + t22 * tmp2

    aPrime = (fPrime1 + i * T2 * f1) * np.exp(i * xi * T2)
    return aPrime


In [43]:
class Potential:
    def __init__(self, T1: float, T2: float, q_func):
        self._T1 = T1
        self._T2 = T2
        self._q_func = q_func

    def T1(self):
        return self._T1

    def T2(self):
        return self._T2

    def q(self, t: float) -> complex:
        return self._q_func(t)

In [54]:
def a_exact_SY(xi, A):
    """
    Аналитическое решение для коэфицента a(xi), потенциала вида A*sech(t). 

    Satsuma-Yajima potential
    """
    return (gamma(0.5 - 1j * xi)**2) / (gamma(0.5 - 1j * xi + A) * gamma(0.5 - 1j * xi - A))

def b_exact_SY(xi, A):
    """
    Аналитическое решение для коэфицента b(xi), потенциала вида A*sech(t). 

    Satsuma-Yajima potential
    """
    return -np.sin(np.pi * A) / np.cosh(np.pi * xi)


n = 10000
xi = 1.0
A = 2.0
t1 = -30
t2 = 30

signal = Potential(
    T1=t1,
    T2=t2,
    q_func=lambda t: A / np.cosh(t) + 0j
)

a_num, b_num = BOsborne(signal, xi, n)
a_prime_num = BOsbornePrime(signal, xi, n)

a_an = a_exact_SY(xi, A)
b_an = b_exact_SY(xi, A)

print("error a =", abs(a_num - a_an))
print("error b =", abs(b_num - b_an))

error a = 1.4276970335474336e-06
error b = 2.1680976161082132e-07


In [55]:
nt = np.array([2**i for i in range(7,16)])

xi = 1.0

a_an = a_exact_SY(xi, A)
b_an = b_exact_SY(xi, A)

arr_errors_a = []
arr_errors_b = []

for n in nt:
    print(n)
    a_num, b_num = BOsborne(signal, xi, n)
    a_prime_num = BOsbornePrime(signal, xi, n)
    arr_errors_a.append(abs(a_num - a_an))
    arr_errors_b.append(abs(b_num - b_an))

128
256
512
1024
2048
4096
8192
16384
32768


In [56]:
np.log2(np.array(arr_errors_a[:-1]) / np.array(arr_errors_a[1:]))

array([2.02503137, 2.00565002, 2.00137701, 2.00034211, 2.00008539,
       2.00002134, 2.00000531, 2.00000123])

In [57]:
np.log2(np.array(arr_errors_b[:-1]) / np.array(arr_errors_b[1:]))

array([1.91403453, 1.98395432, 1.99604392, 1.99901438, 1.99975391,
       1.9999389 , 1.99998638, 2.00000326])

In [60]:
nxi = 1025
arr_xi = np.linspace(-20, 20, nxi)

def fi0(xi):
    if np.abs(a_exact_SY(xi, A)) > 1:
        return a_exact_SY(xi, A)
    else:
        return 1

def NMSE(nxi, arr_xi, nt):
    res = 0
    for i in range(nxi):
        res += np.abs(BOsborne(signal, arr_xi[i], nt) - a_exact_SY(arr_xi[i], A))**2 / np.abs(fi0(arr_xi[i]))**2 
    return (1/n) * res

NMSE(nxi, arr_xi, 1000)

array([1.35078122e-10, 3.12804555e-02])

In [61]:
BOsborne(signal, 1, 1000)

((-0.9691956042745474-0.24629226578546892j),
 (-2.1660500225759282e-05-3.906266078017393e-16j))